# Technology cost sensitivity plot

In [ ]:
import logging
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.WARNING)


In [ ]:
# SETTINGS

labels = ['BASE', 'BASE NEW COST', 'SC', 'SC NEW COST', 'GC', 'GC NEW COST']
scenarios=['base_2025_correct', 'base_cost', 'sudden_2025_correct', 'sudden_cost', 'gradual_2025_correct', 'gradual_cost']

base_file_name = 'base_2025_correct'
sudden_file_name = 'sudden_2025_correct'
gradual_file_name = 'gradual_2025_correct'
base_cost_file_name = 'base_cost'
sudden_cost_file_name = 'sudden_cost'
gradual_cost_file_name = 'gradual_cost'

path = '../result_data/'
fig_path = '../figures/gas/'

years = [2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040]
first_year = 2024
last_year = 2040

In [ ]:
# COLORS

red1 = '#891D2D'
red2 = '#BA3B31'
orange = '#F58221'
yellow = '#FCAF19'
brown = '#440A15'
brown2 = '#B45419'
purple1 = '#3B1053'
purple2 = '#76518E'
purple3 = '#B69DC7'
teal1 = '#032838'
teal2 = '#154655'
teal3 = '#527D77'
teal4 = '#8DB5AF'
teal1 = '#294839'
green1 = '#6DA08C'
green2 = '#6E966E'
green3 = '#A3BDA3'
beige1 = '#7A693B'
beige2 = '#A89677'
beige3 = '#D2CDAD'
grey1 = '#E7E7E7'
grey2 = '#D7D7D7'
grey3 = '#C6C6C6'
grey4 = '#939393'
blue1 = '#3EA1C0'

In [ ]:
# Helper functions

def get_colors(carriers):
    colors = [beige2, beige3, teal3, beige1, teal4, yellow, teal2, brown, brown, brown2, grey4, grey1]
    names = ['CCGT',    'OCGT',  'Biomass',   'Oil',  'Wind',  'Solar'  ,'Hydro', 'Battery', 'Nbattery', 'Geothermal', 'Lost load', 'Demand']
    color_dict = dict(zip(names, colors))
    colors_new = [color_dict[carrier] for carrier in carriers]
    return colors_new

In [ ]:
# PLOT NPV AND SUBSIDIES

yearly_costs_all = pd.read_csv(path + 'total_costs.csv')
capcost = pd.read_csv(path + 'capcost.csv')

yearly_costs = yearly_costs_all[scenarios]
total_costs = yearly_costs.sum(axis=0) /1000
total_costs.columns = labels

costs_dict = {label: total_costs[i] for i, label in enumerate(yearly_costs.columns)}
costs_df = pd.DataFrame([costs_dict])

subsidies_all = pd.read_csv(path + 'subsidies.csv')
subsidies = subsidies_all[scenarios]
subsidies = subsidies / 1000
print(subsidies)
#subsidies.columns = labels

# Create a new figure and axis for the plot
fig, ax = plt.subplots(figsize=(12, 4))

# The x-axis positions for each bar
index = np.arange(len(costs_df.columns))
colors = [brown2, brown2, beige2, beige2, beige3, beige3]

total_costs_bars = ax.bar(index, costs_df.iloc[0], label='NPV', color=colors, edgecolor=colors)

fill_colors_with_alpha = [mcolors.to_rgba(color, alpha=0.4) for color in colors]  # Fill colors with transparency
solid_edge_colors = [mcolors.to_rgba(color, alpha=1) for color in colors]  # Edge colors without transparency



# Plotting the subsidies bars on top of the total costs (NPV)
subsidies_bars = ax.bar(index, subsidies.iloc[0], label='Subsidies', bottom=costs_df.iloc[0], 
                        color=fill_colors_with_alpha, edgecolor=solid_edge_colors, linestyle='--', linewidth=2)

ax.set_xticks(index, fontsize=14)
ax.set_xticklabels(labels, rotation=0)

# # Set the y-axis label
npv_patch = mpatches.Patch(color='darkgrey', label='NPV')
subsidies_patch = mpatches.Patch(facecolor=grey1, edgecolor='darkgrey', linestyle='--', linewidth=1, label='Subsidies')

# Create the legend with custom patches
plt.legend(handles=[subsidies_patch, npv_patch], loc='upper center', edgecolor=grey1, bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=14)
plt.ylabel('Cost [billion €]', fontsize=18)


# Add a title and a legend

plt.ylim(0,12)
plt.grid(axis='y', color=grey1)

# Adjust layout and show the plot
plt.tight_layout()
plt.savefig(fig_path + 'costs.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT PRODUCTION

production_base = pd.read_csv(path + base_file_name + '_production.csv', index_col=0)
production_sudden = pd.read_csv(path + sudden_file_name + '_production.csv', index_col=0)
production_gradual = pd.read_csv(path + gradual_file_name + '_production.csv', index_col=0)
production_base_cap = pd.read_csv(path + base_cost_file_name + '_production.csv', index_col=0)
production_sudden_cap = pd.read_csv(path + sudden_cost_file_name + '_production.csv', index_col=0)
production_gradual_cap = pd.read_csv(path + gradual_cost_file_name + '_production.csv', index_col=0)

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True,
    gridspec_kw={'wspace': 0.1, 'hspace': 0.3})  # Adjust figsize as needed

# Flatten the 2D array of axes to 1D for easier iteration
axes = axes.flatten()

# Define your scenarios
scenarios = [
    (production_base, 'BASE'),
    (production_base_cap, 'BASE NEW COST'),
    (production_sudden, 'SC'),
    (production_sudden_cap, 'SC NEW COST'),
    (production_gradual, 'GC'),
    (production_gradual_cap, 'GC NEW COST')
]


# Loop through the scenarios to create each subplot
for ax, (production_data, title) in zip(axes, scenarios):
    # Ensure production_data is a DataFrame and not a numpy array
    if isinstance(production_data, pd.DataFrame):
        production_data.index = production_data.index.astype(str).str.strip().astype(int)

        ax.stackplot(production_data.index, production_data.T, 
                     colors=get_colors(production_data.columns), labels=production_data.columns)
        ax.set_ylim(0, 22)
        #ax.set_xlim(first_year, last_year)
        ax.set_title(title, fontsize=18)
        # rotate x-tick labels for better readability including all years
        ax.set_xlim(production_data.index.min(), production_data.index.max())
        xticks = production_data.index[::5]  # hvert 5. element i indeksen
        ax.set_xticks(xticks + 1)
        ax.set_xticklabels(xticks + 1, fontsize=18)
        #ax.tick_params(axis='both', which='major', labelsize=14) 
        ax.tick_params(axis='y', labelsize=18)
    else:
        print(f"Data for {title} is not in DataFrame format.")

# Fjern y-ticks i høyre kolonne
for i, ax in enumerate(axes):
    if i % 2 == 1:  # høyre kolonne
        ax.set_yticklabels([])
        ax.set_ylabel('')

# Set the shared x and y-axis labels
fig.text(0.5, 0.04, 'Year', ha='center', fontsize=18)
fig.text(0.04, 0.5, 'Power Generation [TWh]', va='center', rotation='vertical', fontsize=18)

colors=[beige2,beige3,beige1,brown2, teal2, teal4,yellow,teal3, brown, grey4, grey1]
legend = [label for label in production_base.columns if label != 'Nbattery']  # Exclude 'Nbattery'
patches = [mpatches.Patch(color=color, label=label) for label, color in zip(legend, colors) if label != 'Nbattery']
handles = [mpatches.Patch(color=color, label=label) for label, color in zip(legend, colors)]
fig.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, 0), ncol=4, fontsize=14)

#plt.subplots_adjust(hspace=0.3, bottom=0.15)

# Save the figure
plt.savefig(fig_path + 'power_prod.png', dpi=300, bbox_inches='tight') 
plt.show()